# Stage 6: public 7.x visible-sample check (Colab helper)

This Colab helper downloads and runs Kaito Fukami's public `rogii-public-7-061-exact-reproduction` kernel against the three visible sample wells. It is useful for inspecting the code and runtime, but **its CSV cannot be submitted to this Code Competition**. Formal scoring requires copying, committing, and submitting the source notebook on Kaggle, where Kaggle replaces the visible sample with the hidden test set.

The source title reports 7.061; our 2026-07-19 Kaggle page snapshot displayed 7.099. These are scores from Kaggle's hidden-test rerun, not scores produced by this Colab helper and not honest 773-well OOF. Stage 4 OOF remains the leakage-controlled reference until the strong components are independently cross-validated.

Before running, put `kaggle.json` in `MyDrive/kaggle/kaggle.json`. Kaggle competition rules must already be accepted by your account. Do not paste the token into a cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import pandas as pd

REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
PUBLIC_KERNEL = 'kaitofukami/rogii-public-7-061-exact-reproduction'
repo_dir = Path('/content/ROGII')
local_data = Path('/content/rogii-data')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
drive_data = drive_root / 'data'
stage6_root = Path('/content/stage6-public7')
stage6_root.mkdir(parents=True, exist_ok=True)

if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
assert (drive_data / 'train').is_dir(), f'Missing: {drive_data / "train"}'
assert (drive_data / 'test').is_dir(), f'Missing: {drive_data / "test"}'
assert (drive_data / 'sample_submission.csv').is_file(), 'Missing sample_submission.csv'
if not ((local_data / 'train').is_dir() and (local_data / 'test').is_dir()):
    shutil.copytree(drive_data, local_data, dirs_exist_ok=True)
print('repository:', subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip())
print('data:', local_data)

In [ ]:
# Install only the runtime packages used by the public kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kaggle>=2.2,<3', 'lightgbm>=4,<5', 'catboost>=1.2,<2'], check=True)

credential_source = Path('/content/drive/MyDrive/kaggle/kaggle.json')
credential_target = Path.home() / '.kaggle' / 'kaggle.json'
if credential_source.is_file():
    credential_target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(credential_source, credential_target)
    credential_target.chmod(0o600)
elif not (os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY')):
    raise FileNotFoundError('Put kaggle.json in MyDrive/kaggle/kaggle.json, then rerun this cell')
subprocess.run(['kaggle', '--version'], check=True)

In [ ]:
# Download the public kernel source and the exact public artifact packages it references.
kernel_dir = stage6_root / 'kernel'
kernel_dir.mkdir(parents=True, exist_ok=True)
if not list(kernel_dir.glob('*.ipynb')):
    subprocess.run(['kaggle', 'kernels', 'pull', '-p', str(kernel_dir), PUBLIC_KERNEL, '-m'], check=True)

datasets = {
    'ridge': 'ravaghi/wellbore-geology-prediction-artifacts',
    'learned': 'fleongg/rogii-claude-models-pub',
    'model_package': 'pilkwang/rogii-model-package',
}
asset_dirs = {}
for name, handle in datasets.items():
    destination = stage6_root / 'assets' / name
    destination.mkdir(parents=True, exist_ok=True)
    marker = destination / '.download-complete'
    if not marker.is_file():
        subprocess.run(['kaggle', 'datasets', 'download', '-d', handle, '-p', str(destination), '--unzip'], check=True)
        marker.write_text(handle + '\n')
    asset_dirs[name] = destination
    print(name, destination, sum(1 for p in destination.rglob('*') if p.is_file()), 'files')

In [ ]:
# Recreate Kaggle's read-only input layout with links; no competition or artifact data is copied again.
def ensure_link(link: Path, target: Path) -> None:
    target = target.resolve()
    if link.is_symlink():
        assert link.resolve() == target, f'Unexpected existing link: {link}'
        return
    if link.exists():
        raise FileExistsError(f'Refusing to replace existing path: {link}')
    link.parent.mkdir(parents=True, exist_ok=True)
    link.symlink_to(target, target_is_directory=True)

ensure_link(Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'), local_data)
ensure_link(Path('/kaggle/input/datasets/ravaghi/wellbore-geology-prediction-artifacts'), asset_dirs['ridge'])
ensure_link(Path('/kaggle/input/datasets/fleongg/rogii-claude-models-pub'), asset_dirs['learned'])
ensure_link(Path('/kaggle/input/datasets/pilkwang/rogii-model-package'), asset_dirs['model_package'])
Path('/kaggle/working').mkdir(parents=True, exist_ok=True)
public_notebook = next(kernel_dir.glob('*.ipynb'))
source_sha256 = hashlib.sha256(public_notebook.read_bytes()).hexdigest()
print('public notebook:', public_notebook)
print('source sha256:', source_sha256)

In [ ]:
# This is the long-running cell. A T4 GPU is optional; PF work is mainly CPU-bound.
os.chdir('/kaggle/working')
get_ipython().run_line_magic('run', str(public_notebook))

In [ ]:
# Validate and persist the visible-sample output for inspection only. It cannot be submitted.
working_dir = Path('/kaggle/working')
candidate_path = working_dir / 'submission.csv'
assert candidate_path.is_file(), 'The public kernel did not produce /kaggle/working/submission.csv'
sample = pd.read_csv(local_data / 'sample_submission.csv')
candidate = pd.read_csv(candidate_path)
assert list(candidate.columns) == ['id', 'tvt']
assert candidate['id'].tolist() == sample['id'].tolist(), 'Submission IDs/order do not match sample_submission.csv'
assert candidate['id'].is_unique and candidate['tvt'].notna().all()

output_dir = drive_root / 'visible_sample_checks' / 'stage6_public7_v001'
output_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidate_path, output_dir / 'submission.csv')
for diagnostic in working_dir.glob('*.csv'):
    if diagnostic.name != 'submission.csv':
        shutil.copy2(diagnostic, output_dir / diagnostic.name)
report = {
    'kind': 'visible_sample_check_not_submittable',
    'source_kernel': PUBLIC_KERNEL,
    'source_sha256': source_sha256,
    'title_score': 7.061,
    'page_snapshot_score_2026_07_19': 7.099,
    'n_rows': int(len(candidate)),
    'id_order_matches_sample': True,
    'submission_sha256': hashlib.sha256(candidate_path.read_bytes()).hexdigest(),
    'tvt_min': float(candidate['tvt'].min()),
    'tvt_max': float(candidate['tvt'].max()),
}
(output_dir / 'stage6_report.json').write_text(json.dumps(report, indent=2) + '\n')
print('saved:', output_dir / 'submission.csv')
report

## Formal Kaggle submission

Do not upload this CSV. Open the [source Kaggle notebook](https://www.kaggle.com/code/kaitofukami/rogii-public-7-061-exact-reproduction), choose **Copy & Edit**, verify that its competition and dataset Inputs remain attached, turn Internet off, then use **Save Version → Save & Run All**. After the committed run completes, select `/kaggle/working/submission.csv` in Output and click **Submit to Competition**. Kaggle then reruns the notebook on the hidden test set. A 7.x score completes the public positive-control gate; it does not promote overlap-derived target transfer or prove private-leaderboard generalization.